In [1]:
from torch import nn

import torch

import numpy as np

from copy import deepcopy

In [2]:
torch.manual_seed(1)

x = torch.randn(10, 32, 5, 5)

In [3]:
torch.manual_seed(1)


bn = nn.BatchNorm2d(num_features=10, affine=True)

bn.weight = nn.Parameter(torch.rand_like(bn.weight))
bn.bias = nn.Parameter(torch.rand_like(bn.bias))

model = nn.Sequential(
    nn.Conv2d(32, 10, kernel_size=3),
    nn.BatchNorm2d(num_features=10, affine=True),
    nn.Conv2d(10, 10, kernel_size=1),
#     nn.Sigmoid()
)

# train:
model(x)

tensor([[[[-4.7651e-01, -1.8479e-01,  5.6382e-01],
          [ 2.1669e-02,  1.3441e-01,  3.3952e-01],
          [ 6.2893e-01, -3.8615e-01, -5.0052e-01]],

         [[-4.6640e-01, -5.6075e-01, -1.0687e+00],
          [ 4.0206e-02, -1.4978e+00, -5.6955e-01],
          [ 1.3363e-01, -7.4812e-01, -3.9570e-01]],

         [[ 8.0244e-02,  1.7706e-01, -6.9603e-01],
          [ 1.5524e-01, -3.3711e-02, -1.0757e-01],
          [-1.7330e-02, -9.0184e-02, -4.4446e-02]],

         [[-8.1912e-01,  1.8261e-01, -6.2772e-01],
          [ 2.7126e-01, -3.8327e-01, -2.9966e-01],
          [-3.6232e-01,  4.7625e-01,  1.2464e-02]],

         [[-7.0363e-02,  7.5828e-01, -1.5196e-01],
          [ 8.9818e-01, -4.2782e-01,  4.8000e-01],
          [-5.9474e-01,  5.2631e-01,  3.4453e-01]],

         [[ 1.3860e-01, -1.8546e-01,  4.0832e-01],
          [ 1.3025e+00,  3.8013e-01,  1.0905e+00],
          [ 8.8690e-01, -6.1459e-01,  4.8888e-01]],

         [[ 1.9182e-01, -2.6052e-01,  1.5696e-01],
          [ 1.2002e

In [6]:
def merge_convKxK_with_conv1x1(convK, conv1):
    Wk = getattr(convK, "weight")
    bk = getattr(convK, "bias")
    
    W1 = getattr(conv1, "weight")
    
    np.testing.assert_equal(W1.shape[2:], (1, 1))
    W1 = W1.squeeze()
    
    b1 = getattr(conv1, "bias")
    
    Wh = torch.einsum(
        "jiwh,kj->kiwh", 
        Wk, W1
    )
    
    bh = W1 @ bk  + b1
    
    merged_conv = deepcopy(convK)
    merged_conv.weight = nn.Parameter(Wh)
    merged_conv.bias = nn.Parameter(bh)
    
    return merged_conv
    

def merge_model(conv1, bn, conv2):
    
    
    bn_mean = bn.running_mean.clone()

    bn_scale = bn.weight.clone()
    bn_shift = bn.bias.clone()
    bn_std = (bn.running_var.clone() + bn.eps) ** 0.5
    
    W_bn = torch.diag(bn_scale / bn_std)
    
    b_bn = - (bn_scale / bn_std ) * bn_mean + bn_shift
    
    d = b_bn.shape[0]
    
    conv_bn = nn.Conv2d(
        d, d, kernel_size=1
    )
    
    assert hasattr(conv_bn, "weight")
    assert hasattr(conv_bn, "bias")

    conv_bn.weight = nn.Parameter(
        W_bn.unsqueeze(2).unsqueeze(3)
    )
    
    conv_bn.bias = nn.Parameter(b_bn)
        
    return merge_convKxK_with_conv1x1(
        merge_convKxK_with_conv1x1(conv1, conv_bn),
        conv2
    )

model.eval()

torch.manual_seed(2)

x2 = torch.randn(20, 32, 5, 5)

merged_model = merge_model(
    model[0], model[1], model[2]
)

with torch.no_grad():
    expected = model(x2)
    actual = nn.Sequential(
        merged_model,
    )(x2)
    
    
    np.testing.assert_allclose(actual, expected, atol=1e-6)

In [5]:
merged_model

Conv2d(32, 10, kernel_size=(3, 3), stride=(1, 1))